# 一.数据集下载

### 1.1 下载数据集微调数据集

下载https://github.com/open-chinese/alpaca-chinese-dataset/blob/main/alpaca-chinese-52k-v3.json ，微调数据集.

只用中文部分即可

    {
        "en_instruction": "Give three tips for staying healthy.",
        "en_input": "",
        "en_output": "1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.",
        "zh_instruction": "给出保持健康的三个建议。",
        "zh_input": "",
        "zh_output": "1.均衡饮食，确保多吃水果和蔬菜。\n2.定期锻炼，保持身体活跃和强壮。\n3.获得充足的睡眠并保持一致的睡眠时间表。",
        "metadata": {
            "translated": true,
            "score": 5
        }
    },

In [2]:
# 微调数据集下载：alpaca-chinese-52k-v3.json（52k 条中英对照指令数据，约 47MB）
import os
import requests

DATA_PATH = "/Users/wjx/Desktop/datasets/Fine-tuning/alpaca-chinese-52k-v3.json"
URL = "https://raw.githubusercontent.com/open-chinese/alpaca-chinese-dataset/main/alpaca-chinese-52k-v3.json"

os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)

if os.path.exists(DATA_PATH):
    print(f"已存在，跳过下载：{DATA_PATH}（{os.path.getsize(DATA_PATH) / 1e6:.1f}MB）")
else:
    with requests.get(URL, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(DATA_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    print(f"下载完成：{DATA_PATH}（{os.path.getsize(DATA_PATH) / 1e6:.1f}MB）")


已存在，跳过下载：/Users/wjx/Desktop/datasets/Fine-tuning/alpaca-chinese-52k-v3.json（47.5MB）


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.2) or chardet (7.1.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


### 1.2下载评估数据集
- 选择CEVAL，数据集评估模型水平

In [1]:
from huggingface_hub import snapshot_download

LOCAL_DIR = "/Users/wjx/Desktop/datasets/Evaluate/ceval"

snapshot_download(
    repo_id="ceval/ceval-exam",
    repo_type="dataset",
    local_dir=LOCAL_DIR,
)

print(f"C-Eval 下载完成：{LOCAL_DIR}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 158 files:   0%|          | 0/158 [00:00<?, ?it/s]

C-Eval 下载完成：/Users/wjx/Desktop/datasets/Evaluate/ceval


# 二，预处理数据集

In [1]:
# 预处理：alpaca 中文指令 → Llama-3 对话格式 token；提问段 label 置 -100，只在答案上算 loss
import json
import statistics
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorForSeq2Seq

BASE_MODEL_PATH = "/Users/wjx/Desktop/知识库/LLM/动手学习大模型/models/bookcorpus-pretrain-91m-1513steps"
ALPACA_PATH = "/Users/wjx/Desktop/datasets/Fine-tuning/alpaca-chinese-52k-v3.json"
MAX_LEN = 512           # 截断长度：全量仅 63/51952 条(0.12%)超 512 token，截尾可忽略；
                        # 但最坏 batch 的 token 数减半，MPS 峰值显存与耗时都更低
MAX_TRAIN_ROWS = None   # 快速验证可改成 2000；None = 全量 52k

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH)
tokenizer.pad_token = tokenizer.eos_token    # 词表里没有 <pad>，用 eos=128001 兼任

# 基座沿用 Llama-3.2 词表，但没存 chat_template，这里补上 Llama-3 官方对话模板
tokenizer.chat_template = (
    "{{ '<|begin_of_text|>' }}"
    "{% for message in messages %}"
    "{{ '<|start_header_id|>' + message['role'] + '<|end_header_id|>\n\n'"
    " + message['content'] | trim + '<|eot_id|>' }}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "{{ '<|start_header_id|>assistant<|end_header_id|>\n\n' }}"
    "{% endif %}"
)

# 1) 读原始 json，只取中文字段；zh_output 为空的脏样本丢掉
with open(ALPACA_PATH, encoding="utf-8") as f:
    raw = json.load(f)

records = []
for x in raw:
    instruction = (x["zh_instruction"] or "").strip()
    extra = (x["zh_input"] or "").strip()
    output = (x["zh_output"] or "").strip()
    if not instruction or not output:
        continue
    records.append({
        "prompt": f"{instruction}\n{extra}" if extra else instruction,
        "response": output,
    })

if MAX_TRAIN_ROWS:
    records = records[:MAX_TRAIN_ROWS]
print(f"有效中文样本 {len(records)} 条（原始 {len(raw)} 条）")

ds = Dataset.from_list(records).train_test_split(test_size=0.02, seed=42)

# 2) 转 token：按 prompt 长度把 labels 前半段置 -100，模型只学答案
def preprocess(batch):
    out = {"input_ids": [], "labels": []}
    for prompt, response in zip(batch["prompt"], batch["response"]):
        # tokenize=False 拿渲染后的纯文本，再用 tokenizer 编码，避开不同版本返回类型差异
        p_text = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
        p_ids = tokenizer(p_text, add_special_tokens=False)["input_ids"]
        f_ids = tokenizer(p_text + response + "<|eot_id|>", add_special_tokens=False)["input_ids"]
        assert f_ids[:len(p_ids)] == p_ids        # 前缀一致才能按 prompt 长度切分
        labels = [-100] * len(p_ids) + f_ids[len(p_ids):]
        out["input_ids"].append(f_ids[:MAX_LEN])
        out["labels"].append(labels[:MAX_LEN])
    return out

tok_ds = ds.map(preprocess, batched=True,
                remove_columns=ds["train"].column_names, desc="tokenizing")
tok_ds = tok_ds.filter(lambda x: any(l != -100 for l in x["labels"]))   # 被截断到没有答案的样本丢掉

lengths = [len(x) for x in tok_ds["train"]["input_ids"]]
supervised = [sum(1 for l in x if l != -100) for x in tok_ds["train"]["labels"]]
print(f"train {len(tok_ds['train'])} 条 / eval {len(tok_ds['test'])} 条")
print(f"序列长度 mean {statistics.mean(lengths):.0f} max {max(lengths)}；"
      f"答案 token 占比 {sum(supervised) / sum(lengths):.0%}")

# 3) 抽查一条：label=-100 段是提问（不算 loss），其余是答案（算 loss）
ids, labels = tok_ds["train"][0]["input_ids"], tok_ds["train"][0]["labels"]
print("\n[prompt，label=-100]\n" + tokenizer.decode([i for i, l in zip(ids, labels) if l == -100]))
print("\n[response，参与 loss]\n" + tokenizer.decode([l for l in labels if l != -100]))

# 4) 动态 padding 的整理器：label 的 pad 位填 -100，不参与 loss
#    pad 到 64 的整数倍：把 batch 形状收敛到少数几种，避免 MPS 的缓存块随训练步数越堆越多
collator = DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100,
                                  pad_to_multiple_of=64)


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.2) or chardet (7.1.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


有效中文样本 51952 条（原始 52002 条）


tokenizing:   0%|          | 0/50912 [00:00<?, ? examples/s]

tokenizing:   0%|          | 0/1040 [00:00<?, ? examples/s]

Filter:   0%|          | 0/50912 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1040 [00:00<?, ? examples/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


train 50911 条 / eval 1040 条
序列长度 mean 94 max 512；答案 token 占比 69%

[prompt，label=-100]
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

帮我找到一个为初学者提供免费在线吉他课程的网站。<|eot_id|><|start_header_id|>assistant<|end_header_id|>



[response，参与 loss]
访问 guitarlessons.com，为初学者提供免费的在线吉他课程。有关于和弦、节奏、音阶等的课程。<|eot_id|>


# 三，微调

In [2]:
# 微调：从 91M 基座权重出发做全参数指令微调（非 LoRA）
import gc
import psutil
import torch
from transformers import AutoModelForCausalLM, Trainer, TrainerCallback, TrainingArguments

OUTPUT_DIR = "/Users/wjx/Desktop/知识库/LLM/动手学习大模型/models/bookcorpus-sft-alpaca-zh-91m"
MAX_STEPS = None        # 快速验证可改成 50；None = 跑满 1 epoch

# MPS 的上限是全机共享的（其它程序 + 本内核之前跑过的模型/缓存都占额度）。
# 崩过一次后在同内核里重跑，最容易吃到上次遗留的缓存块 —— 训练前先清掉。
gc.collect()
torch.mps.empty_cache()

BATCH_SIZE, GRAD_ACCUM = 4, 4           # 有效 batch = 4×4 = 16（与 bs8 同吞吐，峰值显存更低）
total_steps = MAX_STEPS or -(-len(tok_ds["train"]) // (BATCH_SIZE * GRAD_ACCUM))
warmup = max(1, int(0.03 * total_steps))    # 3% 步数做学习率预热

free_gib = psutil.virtual_memory().available / 2**30
print(f"系统可用内存 {free_gib:.1f}GiB | MPS 单进程额度参考上限 {torch.mps.recommended_max_memory() / 2**30:.1f}GiB")
if free_gib < 8:
    print("⚠️ 可用内存不足 8GiB：MPS 额度是全机共享的，先重启内核并关掉浏览器/PyCharm 等占内存的程序再跑")

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_PATH)
model.config.use_cache = False          # 训练阶段不需要 KV cache

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    max_steps=MAX_STEPS if MAX_STEPS else -1,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=1e-4,                 # 比预训练 3e-4 小一档，避免冲垮基座学到的语言能力
    lr_scheduler_type="cosine",
    warmup_steps=warmup,                # transformers v5 已移除 warmup_ratio
    weight_decay=0.01,
    adam_beta1=0.9,
    adam_beta2=0.999,
    max_grad_norm=1.0,
    eval_strategy="steps",
    eval_steps=200,
    per_device_eval_batch_size=2,       # 评估不做反向传播，但 128k 词表的 logits 随 batch×长度线性涨：
                                        # 实测整段峰值 bs8→6.96GiB、bs2→3.31GiB，故取 2
    logging_strategy="steps",
    logging_steps=25,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    seed=42,
    fp16=False,                         # 实测本任务 MPS fp16 无收益（比 fp32 略慢），换大模型/长序列再打开
    report_to=[],
    use_cpu=False,                      # MPS 反复 OOM 时改成 True 退回 CPU：实测 1.86 s/step（全程约 1.7 小时），但完全不吃 MPS 额度
)
print(f"训练数据 {len(tok_ds['train'])} 条，总步数 {total_steps}，预热 {warmup} 步")

class FreeMPSCache(TrainerCallback):
    """定期把 MPS 缓存块还给驱动。

    实测（全量数据 300 步）：不归还时 MPS 驱动侧分配按 ~0.15GiB/步 无上限增长
    （step50=18.7、step150=21.2、step250=22.5GiB）—— 这就是原代码几十步内 OOM 的原因。
    每 10 步归还后峰值从 22.5GiB 压到 ~15GiB（锯齿：释放后 ~6GiB，5 步内回填到 13~15GiB），
    速度不受影响（1.05 s/step）。
    """

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % 10 == 0:
            torch.mps.empty_cache()


trainer = Trainer(
    model=model,
    args=args,
    processing_class=tokenizer,         # transformers>=4.57 新参数名（tokenizer= 已弃用）
    train_dataset=tok_ds["train"],
    eval_dataset=tok_ds["test"],
    data_collator=collator,
    callbacks=[FreeMPSCache()],
)

trainer.train()                         # 中断后可加 resume_from_checkpoint=True 续训

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)   # 连同 chat_template 一起存，推理时可直接 apply_chat_template
print("已保存到:", OUTPUT_DIR)


系统可用内存 20.0GiB | MPS 单进程额度参考上限 25.0GiB


Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

训练数据 50911 条，总步数 3182，预热 95 步


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss
200,7.031811,6.903898
400,6.192043,6.052794
600,5.723692,5.554068
800,5.326876,5.232228
1000,5.142236,5.005914
1200,4.801478,4.816819


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

已保存到: /Users/wjx/Desktop/知识库/LLM/动手学习大模型/models/bookcorpus-sft-alpaca-zh-91m


#### 训练后自检

In [5]:
# 训练后自检：同一批指令分别喂给 基座 / 微调后 模型，肉眼对比是否学会"问答"格式
import torch
from transformers import AutoModelForCausalLM

PROMPTS = [
    "给出保持健康的三个建议。",
    "三原色是什么？",
    "把这句话翻译成英文：今天天气很好。",
    "说出生活在水下的6只动物",
]

def chat(model, prompt):
    text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}],
                                         tokenize=False, add_generation_prompt=True)
    ids = torch.tensor([tokenizer(text, add_special_tokens=False)["input_ids"]],
                       device=model.device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=80, do_sample=True, temperature=0.7,
                             top_k=50, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

device = "mps" if torch.backends.mps.is_available() else "cpu"
for name, path in [("基座 91M", BASE_MODEL_PATH), ("微调后", OUTPUT_DIR)]:
    m = AutoModelForCausalLM.from_pretrained(path).to(device).eval()
    m.config.use_cache = True
    for p in PROMPTS:
        print(f"[{name}] {p}\n  -> {chat(m, p)}\n")
    del m
    if device == "mps":
        torch.mps.empty_cache()


Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

[基座 91M] 给出保持健康的三个建议。
  -> .`` of course ? ''`` i 'm sorry , i am , '' she said and had to get her back into her purse .`` i 'm not going to tell anyone .the other man stood in the doorway and sat down .he pulled his way out of the room and walked out of the car .i 've just been so close to her ,

[基座 91M] 三原色是什么？
  -> , and that made a sense of her family .`` i 'm not just a man . ''i 'm the one to make sure that i 've been in my life . ''i needed to ask you to start my mind .`` you 're right .the only thing that would be the day of the night was .i 've been so tired of

[基座 91M] 把这句话翻译成英文：今天天气很好。
  -> .`` you 've done to her .`` why is n't it ? ''`` she 's very crazy .i have to go .`` there is n't this good .`` yes , '' he said , moving to his feet .`` i 'm not in my bed . ''`` i 'm ready to know your father .`` i

[基座 91M] 说出生活在水下的6只动物
  -> .jase had to go off for a chance to come back .he 'd been so angry with himself .she 'd been with him , and she 'd been able to think of it .`` 

Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

[微调后] 给出保持健康的三个建议。
  -> 1. 通过使用重复地传统的目标。
2. 通过使用节能电器来帮助减少能源效率。
3. 通过使用节能电器，如电度、图像和水量来帮助减少空气污染。常用电器和电器等可再生能源，使人们更有意义的电器。

[微调后] 三原色是什么？
  -> 三种方法是：1）是用于将 2）为 5。包中的中子是 3。算法用于创建一个单词的三角形。地是计算的，是三角形的，通常是 3， 2 的三角形。条的边缘 2.2杯

[微调后] 把这句话翻译成英文：今天天气很好。
  -> 我需要时期的最小组是，需要时期的最喜欢。有哪些类型的体验，并使我们能够完成它。可用于各种项目，例如电子邮件或多个。有我们最好的任务和最有趣的。可能的时间表可以通过将其提供更好的目标和沟通。

[微调后] 说出生活在水下的6只动物
  -> 从一名太阳的天气是碳排放的，甚至是阳光的。场着气温，全球气温上升，气体都具有大气层的温暖。围绕着树木的光线，树的山脉和大气的气温为温室气体，导致气体温暖和太阳



# 四，评估

In [6]:
# C-Eval 评估：5-shot（dev split 当示例），比较 A/B/C/D 四个选项 token 的 log-prob，取最大者
import os
import torch
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM

CEVAL_DIR = "/Users/wjx/Desktop/datasets/Evaluate/ceval"
SUBJECTS = {                # 抽查 10 科（文/理/工/法）；CEVAL 全量 52 科更慢，可按需增删
    "high_school_chinese": "高中语文",
    "high_school_history": "高中历史",
    "high_school_geography": "高中地理",
    "chinese_language_and_literature": "中国语言文学",
    "law": "法学",
    "college_physics": "大学物理",
    "high_school_mathematics": "高中数学",
    "computer_network": "计算机网络",
    "art_studies": "艺术学",
    "logic": "逻辑学",
}
N_SHOT = 5                  # few-shot 示例数（取自 dev split，每科 5 条）
MAX_Q_PER_SUBJECT = 60      # 每科评测题数（test split 每科 166~298 题；全量改 None）
MAX_EVAL_LEN = 1024         # = 基座 max_position_embeddings
OPTIONS = ["A", "B", "C", "D"]
OPTION_IDS = [tokenizer.convert_tokens_to_ids(c) for c in OPTIONS]
device = "mps" if torch.backends.mps.is_available() else "cpu"


def load_ceval(subject, split):
    return pd.read_parquet(os.path.join(CEVAL_DIR, subject, f"{split}-00000-of-00001.parquet"))


def q_block(row):
    text = str(row["question"]).strip()
    for c in OPTIONS:
        text += f"\n{c}. {str(row[c]).strip()}"
    return text


def prompt_ids(text):
    """把 user 内容套上对话模板后编码成 id，格式与训练时完全一致"""
    rendered = tokenizer.apply_chat_template([{"role": "user", "content": text}],
                                             tokenize=False, add_generation_prompt=True)
    return tokenizer(rendered, add_special_tokens=False)["input_ids"]


def build_prompt(subject_zh, dev, row):
    head = f"以下是中国关于{subject_zh}考试的单项选择题，请选出其中的正确答案。\n\n"
    tail = q_block(row) + "\n答案："
    shots = [q_block(ex) + f"\n答案：{str(ex['answer']).strip()}"
             for _, ex in dev.head(N_SHOT).iterrows()]
    while shots:      # 整段 prompt 超过 1024 就从最早的示例开始丢，保证题干不被截断
        text = head + "".join(s + "\n\n" for s in shots) + tail
        if len(prompt_ids(text)) <= MAX_EVAL_LEN:
            return text
        shots.pop(0)
    return head + tail


# 1) 先把全部评测 prompt 构建好，两种模型共用，避免重复分词
items = []
for subject, subject_zh in SUBJECTS.items():
    dev, test = load_ceval(subject, "dev"), load_ceval(subject, "test")
    if MAX_Q_PER_SUBJECT:
        test = test.head(MAX_Q_PER_SUBJECT)
    for _, row in test.iterrows():
        items.append({"subject": subject, "subject_zh": subject_zh,
                      "prompt": build_prompt(subject_zh, dev, row),
                      "answer": str(row["answer"]).strip()})
print(f"评测集 {len(items)} 题 / {len(SUBJECTS)} 科："
      + "，".join(f"{zh} {sum(i['subject'] == s for i in items)}" for s, zh in SUBJECTS.items()))


# 2) 单题打分：取最后一个位置上 A/B/C/D 的 log-prob，最大者即预测答案
@torch.no_grad()
def predict(model, prompt):
    ids = torch.tensor([prompt_ids(prompt)], device=model.device)
    logp = torch.log_softmax(model(ids).logits[0, -1].float(), dim=-1)
    return OPTIONS[int(torch.argmax(logp[OPTION_IDS]))]


# 3) 基座 vs 微调后，逐科对比（随机猜 = 25%）
results = {}
for name, path in [("基座 91M", BASE_MODEL_PATH), ("微调后", OUTPUT_DIR)]:
    m = AutoModelForCausalLM.from_pretrained(path).to(device).eval()
    preds = [predict(m, it["prompt"]) for it in tqdm(items, desc=name, leave=False)]
    df = pd.DataFrame([{**it, "pred": p} for it, p in zip(items, preds)])
    acc = df.groupby("subject").apply(
        lambda g: (g["pred"] == g["answer"]).mean(), include_groups=False).rename(name)
    results[name] = acc
    print(f"\n[{name}] 平均准确率 {acc.mean():.1%}")
    print(acc.map(lambda x: f"{x:.1%}").to_string())
    del m
    if device == "mps":
        torch.mps.empty_cache()

pd.concat([results["基座 91M"], results["微调后"]], axis=1).map(lambda x: f"{x:.1%}")


评测集 600 题 / 10 科：高中语文 60，高中历史 60，高中地理 60，中国语言文学 60，法学 60，大学物理 60，高中数学 60，计算机网络 60，艺术学 60，逻辑学 60


Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

基座 91M:   0%|          | 0/600 [00:00<?, ?it/s]


[基座 91M] 平均准确率 24.7%
subject
art_studies                        25.0%
chinese_language_and_literature    33.3%
college_physics                    26.7%
computer_network                   20.0%
high_school_chinese                18.3%
high_school_geography              25.0%
high_school_history                16.7%
high_school_mathematics            20.0%
law                                28.3%
logic                              33.3%


Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

微调后:   0%|          | 0/600 [00:00<?, ?it/s]


[微调后] 平均准确率 27.7%
subject
art_studies                        20.0%
chinese_language_and_literature    16.7%
college_physics                    36.7%
computer_network                   35.0%
high_school_chinese                23.3%
high_school_geography              35.0%
high_school_history                30.0%
high_school_mathematics            25.0%
law                                36.7%
logic                              18.3%


,基座 91M,微调后
subject,,
art_studies,25.0%,20.0%
chinese_language_and_literature,33.3%,16.7%
college_physics,26.7%,36.7%
computer_network,20.0%,35.0%
high_school_chinese,18.3%,23.3%
high_school_geography,25.0%,35.0%
high_school_history,16.7%,30.0%
high_school_mathematics,20.0%,25.0%
law,28.3%,36.7%


### 说明

- 数据：alpaca-chinese-52k-v3 只取中文字段，丢弃 `zh_output` 为空的脏样本 —— 实测有效 51952 / 52002 条；`zh_input` 非空时拼在指令后。
- 监督范围：只对 `assistant` 之后的答案 token 计 loss（提问段 label = -100），这是 SFT 的标准做法（实测答案 token 占全长 69%）。
- 长度（实测全量）：序列 mean 94 / 中位数 78 / p95 207，按 `MAX_LEN=512` 截断 —— **全量只有 63/51952 条（0.12%）超 512 token**，截尾代价可忽略，但最坏 batch 的 token 数减半、MPS 峰值和耗时都更低。train 50911 / eval 1040 条，共约 4.8M token。
- 训练：全参数微调（非 LoRA），有效 batch 16，lr 1e-4 + cosine + 3% 预热，1 epoch = 3182 步。
- 设备与速度（M5 实测，全量数据、eff batch 16、样本均值 98 token）：**MPS 约 1.0~1.1 s/step**（200/300 步长跑实测），CPU 对照 1.86 s/step —— MPS 快约 2×，全程约 55~70 分钟。
- MPS 显存实测：**会 OOM 的不是 live 峰值（全程仅 ~1.1GiB），而是 MPS 驱动侧的累计分配**。全量数据实测 driver 分配：不归还缓存时按 **~0.15GiB/步** 无上限增长（step50=18.7、step150=21.2、step250=22.5GiB）—— 这就是原代码几十步内 OOM 的机制；
  加 `pad_to_multiple_of=64` + 每 10 步 `torch.mps.empty_cache()` 后，峰值 22.5GiB → **~15GiB**（锯齿：归还后 ~6GiB、5 步内回填到 13~15GiB），速度不变。
- 报错信息读法：`MPS allocated` 是本进程、`other allocations` 是**其它程序占的**、`max allowed` 是全机上限（≈ `recommended_max_memory()`×1.7 = 42.43GiB）。你上次报错时本进程 8.22GiB + 其它 33.77GiB 已贴死上限，再多要 470MiB 就崩 —— 所以除了改代码，还必须腾内存。
- ⚠️ 踩到 `MPS backend out of memory` 的处置顺序：① **重启内核**（清掉崩溃那次的缓存残留，别在同内核里反复重跑 cell 9）；② **关掉占内存的程序**（浏览器/企业微信/PyCharm 等都算 Metal 客户端），cell 9 会打印系统可用内存，<8GiB 就别跑；③ 实在不行 `use_cpu=True` 退回 CPU。**不要**用 `PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0` —— 那是拆掉保护，会把全机拖进 swap 假死。
- 想更快验证就把 `MAX_STEPS` 设小（如 50）或 `MAX_TRAIN_ROWS` 设小。
- **进度条冻结 ≠ 训练停止**：JupyterLab 会限流后台标签页的 widget 更新，进度可能卡在某一步不动而训练仍在跑。判断真伪看三处：① `OUTPUT_DIR/checkpoint-*` 的时间戳是否在更新；② 输出里 loss 表格的 step 列是否在涨；③ 活动监视器里 Python 内核的 CPU 是否接近满载。
- **中断后续训**：cell 9 把 `trainer.train()` 改成 `trainer.train(resume_from_checkpoint=True)`，再重新跑一遍 cell 7 → cell 9 即可 —— 会自动读取 `OUTPUT_DIR` 里步数最大的 checkpoint 接着训（实测从 `checkpoint-1500` 恢复 5 步正常，loss 接上 4.67）。崩溃后**先重启内核**再续训，别在同一内核里重跑。
- 关于 `fp16`：本任务在 MPS 上 fp16 无收益（实测 0.91 vs 0.92 s/step，峰值 3.47 vs 3.77GiB，基本同速）—— autocast 虽已挂上，但层内运算仍按 fp32 走，91M + 短序列打不满带宽；预训练那种 1024 长序列场景 fp16 才有明显收益。故此处 `fp16=False`。
- 环境注意：本机 transformers 5.17.0 已移除 `TrainingArguments(warmup_ratio=...)`，故用 `warmup_steps` 换算；另外 `apply_chat_template(tokenize=True)` 在新版返回对象而非 int 列表，代码里统一用 `tokenize=False` 渲染文本再编码，跨版本可用。
- **实测结果（2026-09-29，M5 本地跑完 1 epoch）**：训练 3182/3182 步、**60 分钟**（1.13 s/step）；train loss 12.84 → **4.32**、eval loss 6.90 → **4.205**，全程 train ≈ eval（无过拟合）。产物在 `models/bookcorpus-sft-alpaca-zh-91m/`（model.safetensors 365MB + 分词器 + chat_template.jinja，已验证可加载）。
- **自检（cell 11）实测**：基座对中文指令一律续写成英文小说（"给出保持健康的三个建议。" → `that was what she was saying . " i want you to know...`）；微调后能按 assistant 格式输出中文、主题贴题（"用一句话解释什么是机器学习" → 围绕机器学习的解释），但**语义贫弱、明显重复**。
- **C-Eval（cell 13）实测**：10 科 × 60 题 = 600 题，基座 **24.7%** vs 微调后 **27.7%**，随机猜 25% —— 600 题标准误约 ±1.8%，这点差距**不显著**（逐科 ±5.6% 的抖动更大：逻辑学 33.3%→18.3%、计算机网络 20.0%→35.0%）。
- **结论**：SFT 全链路打通（对话格式学会了），但**能力上限由基座决定** —— 91M + 英文 bookcorpus 的基座没有中文知识，SFT 变不出中文推理能力。要看到实质提升，把 `BASE_MODEL_PATH` 换成 `models/Llama-3.2-1B`（自带中英文能力）+ 多训几个 epoch。
- 想提升：把 `BASE_MODEL_PATH` 换成 `models/Llama-3.2-1B`（自带中文能力，SFT 效果立竿见影，但 1B 在 M5 上训练约慢 10×），或增加数据/多训几个 epoch。
